# 01 — Prepare Adaptive RAG training data

This notebook orchestrates existing `adaptive_rag` functions. **Run stages deliberately**: dataset generation overwrites existing subset Parquet files; teacher generation uses paid API calls; retrieval uses Weaviate and a local SLM. Rerunning teacher generation/retrieval skips completed question IDs.

**Data split policy:** teacher generation, retrieval-derived labeling and SFT export operate on **train only**. Validation/test are held out for later evaluation.
**Configuration:** Adjust `TRAIN_SIZE`, `VAL_SIZE`, `TEST_SIZE`, `TEACHER_PILOT_SIZE`, and `TEACHER_MAX_QUESTIONS` in Section 1. When changing subsets after generating teacher/retrieval results, use fresh output files to avoid mixing runs.


## 0. Environment and project paths

In [ ]:
# Run from a local checkout with the package installed (`pip install -e . --no-deps`),
# or clone/install the repository first in Colab. Do not store API keys in notebooks.
# In Colab, after cloning your repository:
# %cd /content/Adaptive-RAG
# %pip install -e ".[training]"  # or install the appropriate project extras
# %pip install openai pyarrow

from pathlib import Path
import os, json
from collections import Counter
from adaptive_rag.config import TRAIN_SUBSET_PATH, VAL_SUBSET_PATH, TEST_SUBSET_PATH
from adaptive_rag.data.prepare_data import prepare_subsets, load_subset
from adaptive_rag.data.teacher_plans import generate_dataset_plans, load_teacher_plans
from adaptive_rag.evaluation.teacher_retrieval import (
    evaluate_teacher_dataset, load_teacher_retrieval, export_teacher_retrieval_parquet
)
from adaptive_rag.data.training_data import (
    label_retrieval_results, summarize_labels, export_labeled_data,
    export_sft_examples, export_label_summary_parquet
)

PROJECT_ROOT = Path(TRAIN_SUBSET_PATH).parent.parent
OUTPUT_DIR = PROJECT_ROOT / 'subsets' / 'planner_training'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_JSONL = OUTPUT_DIR / 'teacher_plans.jsonl'
RETRIEVAL_JSONL = OUTPUT_DIR / 'teacher_retrieval.jsonl'  # Override for a separate run.
RETRIEVAL_PARQUET = OUTPUT_DIR / 'teacher_retrieval.parquet'
LABEL_JSONL = OUTPUT_DIR / 'planner_labels.jsonl'
SFT_JSONL = OUTPUT_DIR / 'planner_train.jsonl'
print('Project root:', PROJECT_ROOT)
print('Data outputs:', OUTPUT_DIR)

**Colab note:** `config.py` resolves `subsets/` relative to the installed package source. Clone the repository and install it editable in Colab if you want those paths to point to your checkout. Ensure the Weaviate service is reachable from Colab before retrieval.

## 1. Configure and prepare dataset subsets (run once; rebuilding overwrites existing files)


In [ ]:
# Central dataset and teacher-generation settings. Change these values here only.
TRAIN_SIZE = 500
VAL_SIZE = 200
TEST_SIZE = 200
SUBSET_STRATEGY = 'balanced'
TEACHER_PILOT_SIZE = 10
TEACHER_MAX_QUESTIONS = None  # None means all training questions; otherwise an integer limit

for name, size in [('TRAIN_SIZE', TRAIN_SIZE), ('VAL_SIZE', VAL_SIZE), ('TEST_SIZE', TEST_SIZE)]:
    assert isinstance(size, int) and size > 0, f'{name} must be a positive integer'
assert isinstance(TEACHER_PILOT_SIZE, int) and TEACHER_PILOT_SIZE > 0
assert TEACHER_MAX_QUESTIONS is None or (isinstance(TEACHER_MAX_QUESTIONS, int) and TEACHER_MAX_QUESTIONS > 0)

SUBSET_SIZES = {'train': TRAIN_SIZE, 'val': VAL_SIZE, 'test': TEST_SIZE}
subset_config = {split: {'n': size, 'strategy': SUBSET_STRATEGY} for split, size in SUBSET_SIZES.items()}
print('Requested subset sizes:', SUBSET_SIZES)

# Change to True only when ready to replace the existing Parquet subsets.
REBUILD_SUBSETS = False
if REBUILD_SUBSETS:
    prepare_subsets(subset_config)
else:
    print('Dataset rebuilding skipped. Set REBUILD_SUBSETS=True to run.')


In [ ]:
expected = SUBSET_SIZES
paths = {'train': TRAIN_SUBSET_PATH, 'val': VAL_SUBSET_PATH, 'test': TEST_SUBSET_PATH}
subsets = {}
for split, path in paths.items():
    if not Path(path).exists():
        print(f'{split}: missing {path}')
        continue
    ds = load_subset(path)
    subsets[split] = ds
    print(split, len(ds), Counter(ds['source']))
    assert len(set(map(str, ds['id']))) == len(ds), f'Duplicate IDs in {split}'
    assert len(ds) == expected[split], f'Expected {expected[split]} {split} questions'
if len(subsets) == 3:
    ids = {s: set(map(str, ds['id'])) for s,ds in subsets.items()}
    for a,b in [('train','val'),('train','test'),('val','test')]:
        assert not (ids[a] & ids[b]), f'Overlap between {a} and {b}'
    print('All splits have expected sizes and disjoint question IDs.')

**Important:** Check that the Weaviate collection contains the gold passages for the newly sampled questions. The dataset preparation code creates the corpus but does not ingest it. Retrieval recall is not meaningful if the indexed corpus differs from the source corpus.

## 2. Teacher generation — ONCE and SPLIT plans

In [ ]:
# Set a verified API model ID from your OpenAI account/docs.
# Do not assume a display name is the actual API identifier.
TEACHER_MODEL = os.environ.get('OPENAI_TEACHER_MODEL', '')

# Optional: set OPENAI_API_KEY in your environment or Colab secrets.
# This notebook never prints the key.
train_ds = subsets.get('train')
if train_ds is None:
    raise RuntimeError('Prepare and validate train subset first.')
questions = [
    {'question_id': str(row['id']), 'question': str(row['question']), 'dataset': str(row['source'])}
    for row in train_ds
]
print('Questions prepared:', len(questions))
teacher_questions = questions[:TEACHER_MAX_QUESTIONS] if TEACHER_MAX_QUESTIONS is not None else questions
print('Questions selected for teacher generation:', len(teacher_questions))


In [ ]:
# Small paid API pilot first. Re-execution skips already completed IDs.
RUN_TEACHER_PILOT = False
if RUN_TEACHER_PILOT:
    assert TEACHER_MODEL, 'Set OPENAI_TEACHER_MODEL to a verified API model ID'
    assert os.environ.get('OPENAI_API_KEY'), 'Configure OPENAI_API_KEY securely'
    print(generate_dataset_plans(teacher_questions[:TEACHER_PILOT_SIZE], TEACHER_JSONL, model=TEACHER_MODEL))
else:
    print('Pilot disabled.')

In [ ]:
# Inspect pilot before proceeding. Do not include answers or gold evidence in teacher prompts.
plans = load_teacher_plans(TEACHER_JSONL)
print('Saved teacher plans:', len(plans))
for rec in plans[:2]:
    print(json.dumps({k: rec[k] for k in ('question_id','question','once','split')}, indent=2, ensure_ascii=False))

In [ ]:
RUN_TEACHER_FULL = False
if RUN_TEACHER_FULL:
    assert TEACHER_MODEL and os.environ.get('OPENAI_API_KEY')
    print(generate_dataset_plans(teacher_questions, TEACHER_JSONL, model=TEACHER_MODEL))
else:
    print('Full teacher generation disabled.')

In [ ]:
plans = load_teacher_plans(TEACHER_JSONL)
train_ids = {q['question_id'] for q in questions}
plan_ids = {str(p['question_id']) for p in plans}
assert plan_ids <= train_ids, 'Teacher plans include questions outside the training subset'
print(f'Teacher coverage: {len(plan_ids)}/{len(train_ids)}')
print('Missing:', len(train_ids-plan_ids))

## 3. Evaluate paired plans through existing retrieval pipeline

A legacy retrieval JSONL without its companion `*.jsonl.config.json` can still be used for labeling and export if it covers the current training subset. The cell below skips retrieval for a complete legacy file.

For an incomplete legacy file, the default output switches to `teacher_retrieval_v2.jsonl`. The old file stays in place. Enable `RUN_RETRIEVAL` to rebuild retrieval using the existing `teacher_plans.jsonl`; leave the teacher-generation switches off. Keep the new JSONL and its companion config together to resume later. If the new path already belongs to a different run, set a fresh `RETRIEVAL_JSONL` in Section 0 and rerun the notebook setup.

In [ ]:
# Uses existing Weaviate collection, embedding model, reranker and Qwen rewriting model.
# Can take substantial time. Completed questions are skipped on rerun.
RUN_RETRIEVAL = False
legacy_complete = False
manifest = RETRIEVAL_JSONL.with_suffix(RETRIEVAL_JSONL.suffix + '.config.json')
if RETRIEVAL_JSONL.exists() and not manifest.exists():
    training_rows = {str(row['id']): row for row in train_ds}
    legacy_rows = load_teacher_retrieval(RETRIEVAL_JSONL)
    legacy_complete = {str(row['question_id']) for row in legacy_rows} == train_ids and all(
        row['question'] == training_rows[str(row['question_id'])]['question']
        and row['gold_passage_ids'] == training_rows[str(row['question_id'])]['gold_passage_ids']
        for row in legacy_rows
    )
    if not legacy_complete:
        legacy_path = RETRIEVAL_JSONL
        replacement = legacy_path.with_name(f'{legacy_path.stem}_v2.jsonl')
        replacement_config = replacement.with_suffix('.jsonl.config.json')
        if replacement.exists() and not replacement_config.exists():
            raise ValueError(f'{replacement} also lacks checkpoint metadata. Set a fresh RETRIEVAL_JSONL in Section 0.')
        RETRIEVAL_JSONL = replacement
        print(f'Keeping {legacy_path.name}. Retrieval output: {RETRIEVAL_JSONL.name}')

RETRIEVAL_PARQUET = RETRIEVAL_JSONL.with_suffix('.parquet')
if legacy_complete:
    print(f'Using complete legacy retrieval results from {RETRIEVAL_JSONL.name}.')
elif RUN_RETRIEVAL:
    assert plan_ids == train_ids, 'Complete teacher generation first'
    print(evaluate_teacher_dataset(TRAIN_SUBSET_PATH, TEACHER_JSONL, RETRIEVAL_JSONL, max_k=7))
else:
    print('Retrieval disabled.')

In [ ]:
retrieval_records = load_teacher_retrieval(RETRIEVAL_JSONL)
retrieval_ids = {str(r['question_id']) for r in retrieval_records}
assert retrieval_ids <= train_ids, 'Retrieval results include questions outside training subset'
print(f'Retrieval coverage: {len(retrieval_ids)}/{len(train_ids)}')
if retrieval_records:
    for metric in ['recall','ndcg']:
        for strategy in ['once','split']:
            key = f'{strategy}_{metric}_at_7'
            print(key, round(sum(float(r[key]) for r in retrieval_records)/len(retrieval_records),4))

## 4. Export immutable retrieval metrics (no retrieval rerun)

In [ ]:
if retrieval_records:
    export_teacher_retrieval_parquet(RETRIEVAL_JSONL, RETRIEVAL_PARQUET)
    print('Saved:', RETRIEVAL_PARQUET)
else:
    print('No results to export yet.')

## 5. Label strategy and export SFT data (offline; repeat freely)

In [ ]:
# Options: recall_at_k, mean_recall, ndcg_at_k, mean_ndcg, combined
LABEL_METRIC = 'mean_recall'
LABEL_K = 7
LABEL_THRESHOLD = 0.1
LABEL_ALPHA = 0.6  # used only for 'combined'

if not retrieval_records:
    print('No retrieval results available for labeling.')
else:
    labels = label_retrieval_results(
        RETRIEVAL_JSONL, metric=LABEL_METRIC, k=LABEL_K,
        threshold=LABEL_THRESHOLD, alpha=LABEL_ALPHA
    )
    print(json.dumps(summarize_labels(labels), indent=2))
    print('Preview:', json.dumps(labels[0], indent=2, ensure_ascii=False)[:2000])

In [ ]:
EXPORT_LABELS = False
if EXPORT_LABELS:
    assert retrieval_ids == train_ids, f'Wait for all {TRAIN_SIZE} retrieval pairs before final export'
    labels = label_retrieval_results(RETRIEVAL_JSONL, metric=LABEL_METRIC,
                                     k=LABEL_K, threshold=LABEL_THRESHOLD, alpha=LABEL_ALPHA)
    export_labeled_data(labels, LABEL_JSONL)
    export_sft_examples(labels, SFT_JSONL)
    export_label_summary_parquet(labels, OUTPUT_DIR / 'label_summary.parquet')
    print('Exported:', LABEL_JSONL, SFT_JSONL)
else:
    print('Final export disabled.')

## 6. Before training

- Inspect SPLIT plans for decomposition quality and label balance by source.
- Record the teacher model ID, prompt version, retrieval configuration, and threshold in your experiment notes.
- Preserve the original teacher JSONL and retrieval JSONL when experimenting with labeling rules.
- `planner_train.jsonl` is the **training source**. Create a separate validation split from the training examples during fine-tuning; do not use held-out test questions for training labels.
- Never commit API keys, private `.env` files, or large model checkpoints.